  #                                                                       # *****DATA PRE PROCESSING*****

# ***Demography Dataset***

In [ ]:
#Importing all the Necessary Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.simplefilter("ignore", UserWarning)

### **1. Read the CSV file and inspect the data. This confirms the file loaded correctly and shows missing values before you change anything**

In [ ]:
# Read the demography CSV File and inspect the data 
dfDEMO = pd.read_csv(r"C:\Users\vella\Desktop\Numpy Ninja\Team3_PyCoders_PythonHackathon_SEP2026\Python_Hackathon_Sep_2026\Python_Hackathon_Sep_2026\cardiac_failure\demography.csv")
print(dfDEMO.shape)       # Rows and columns
print(dfDEMO.head())      # First 5 rows
dfDEMO.info()   
dfDEMO.describe()         
print(dfDEMO.isna().sum())
print("Duplicate patient IDs:", dfDEMO["inpatient_number"].duplicated().sum())


(2009, 7)
   inpatient_number  gender  weight  height        bmi     occupation agecat
0                 5     NaN     NaN     NaN  46.000000            NaN    NaN
1            827040  Female    50.0    1.45  23.781213            NaN  69-79
2            857781    Male    50.0    1.64  18.590125  UrbanResident  69-79
3            743087  Female    51.0    1.63  19.195303  UrbanResident  69-79
4            866418    Male    70.0    1.70  24.221453         farmer  59-69
<class 'pandas.DataFrame'>
RangeIndex: 2009 entries, 0 to 2008
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   inpatient_number  2009 non-null   int64  
 1   gender            2008 non-null   str    
 2   weight            2008 non-null   float64
 3   height            2008 non-null   float64
 4   bmi               2009 non-null   float64
 5   occupation        1981 non-null   str    
 6   agecat            2008 non-null   str    
dtypes: fl

### **2. Removed the empty record. Patient ID 5 has no gender, weight, height,occupation, or age category. Its BMI value alone is not enuogh to use the record for demographic analysis. So this removes one row**


In [212]:
dfDEMO = dfDEMO.dropna(
    subset=["gender", "weight", "height", "occupation", "agecat"],
    how="all"
).copy()

### **3. Removed extra spaces from the text columns so values such as "Male" and " Male " are treated as the same category. I also make the occupation names consistent and easier to read. This prevents one category from appearing under different labels in charts and counts.**

In [213]:
text_columns = ["gender", "occupation", "agecat"]

for column in text_columns:
    dfDEMO[column] = dfDEMO[column].astype("string").str.strip()

dfDEMO["occupation"] = dfDEMO["occupation"].replace({
    "UrbanResident": "Urban Resident",
    "farmer": "Farmer",
    "worker": "Worker"
})

dfDEMO["occupation"] = dfDEMO["occupation"].fillna("Unknown")

### **4. Impossible measurements, Weight values of zero or less and height values below 1.0 are flagged for review. The code marks those rows in measurement and changes the flagged values to missing so they do not affect BMI calculations. It then counts how many records were flagged.**

In [214]:
dfDEMO["measurement"] = pd.Series(False, index=dfDEMO.index)

invalid_weight = dfDEMO["weight"] <= 0
invalid_height = dfDEMO["height"] < 1.0

dfDEMO.loc[invalid_weight | invalid_height, "measurement"] = True

dfDEMO.loc[invalid_weight, "weight"] = np.nan
dfDEMO.loc[invalid_height, "height"] = np.nan

print("Records with measurement issues:", dfDEMO["measurement"].sum())

Records with measurement issues: 7


### **5. Recalculated BMI from Valid measurements. The orginial BMI values match the recorded weights and heights, including the incorrect heights and zero weights, Recalculating after flagging those measurements makes BMI missing for the seven affected records**

In [215]:
dfDEMO["bmi_original"] = dfDEMO["bmi"].round(2)

dfDEMO["bmi"] = dfDEMO["weight"] / (dfDEMO["height"] ** 2)
dfDEMO["bmi"] = dfDEMO["bmi"].round(2)


### **6. Reviewed the cleaned data to make sure it is ready for analysis. I check the number of rows and unique patients, see which values are still missing, review the weight, height, and BMI ranges, and count each category. Finally, I display the records flagged for measurement issues so I can inspect them before using them in calculations.**

In [ ]:
print("Rows:", len(dfDEMO))
print("Unique patient IDs:", dfDEMO["inpatient_number"].nunique())
print("\nMissing values:")
print(dfDEMO.isna().sum())

print("\nNumeric summary:")
print(dfDEMO[["weight", "height", "bmi"]].describe())

print("\nCategory counts:")
for column in ["gender", "occupation", "agecat"]:
    print(f"\n{column}")
    print(dfDEMO[column].value_counts(dropna=False))

print("\nRecords needing measurement review:")
print(
    dfDEMO.loc[
        dfDEMO["measurement"],
        ["inpatient_number", "weight", "height", "bmi_original"]
    ]
)

Rows: 2008
Unique patient IDs: 2008

Missing values:
inpatient_number    0
gender              0
weight              3
height              4
bmi                 7
occupation          0
agecat              0
measurement         0
bmi_original        0
dtype: int64

Numeric summary:
            weight       height          bmi
count  2005.000000  2004.000000  2001.000000
mean     52.562244     1.570110    21.288046
std      10.713048     0.081954     3.827647
min       8.000000     1.200000     3.460000
25%      45.000000     1.500000    18.490000
50%      50.000000     1.560000    20.760000
75%      60.000000     1.620000    23.440000
max     115.000000     1.830000    39.110000

Category counts:

gender
gender
Female    1163
Male       845
Name: count, dtype: Int64

occupation
occupation
Urban Resident    1670
Farmer             198
Others              89
Unknown             27
Worker              17
Officer              7
Name: count, dtype: Int64

agecat
agecat
69-79     715
79-89   

### **7. Save analysis and reviewing the files**

In [217]:
dfDEMO.to_csv(r"C:\Users\vella\Desktop\Numpy Ninja\Team3_PyCoders_PythonHackathon_SEP2026\Python_Hackathon_Sep_2026\Python_Hackathon_Sep_2026\cardiac_failure\Data_Cleaning\demography_cleaned.csv", index=False)

#                                                          ***patienthistory***

### **1. Load the File and keep the Original**

In [5]:
dfPH = pd.read_csv(r"C:\Users\vella\Desktop\Numpy Ninja\Team3_PyCoders_PythonHackathon_SEP2026\Python_Hackathon_Sep_2026\Python_Hackathon_Sep_2026\cardiac_failure\patienthistory.csv")
df_original = dfPH.copy()
display(dfPH.head())
print("Rows:", dfPH.shape[0])
print("Columns:", dfPH.shape[1])

,inpatient_number,cerebrovascular_disease,dementia,chronic_obstructive_pulmonary_disease,connective_tissue_disease,peptic_ulcer_disease,diabetes,moderate_to_severe_chronic_kidney_disease,hemiplegia,leukemia,malignant_lymphoma,solid_tumor,liver_disease,aids,cci_score,type_ii_respiratory_failure,acute_renal_failure
0,857781,0,0,1,0,0.0,1,0.0,0,0,0,0,0.0,0,2.0,nontypeii,0
1,743087,0,0,0,0,0.0,0,0.0,0,0,0,0,0.0,0,0.0,nontypeii,0
2,866418,0,0,0,0,0.0,0,0.0,0,0,0,0,0.0,0,0.0,nontypeii,0
3,775928,0,0,1,0,0.0,0,1.0,0,0,0,0,0.0,0,2.0,nontypeii,0
4,810128,0,0,0,0,0.0,0,0.0,0,0,0,0,0.0,0,0.0,nontypeii,0


Rows: 2008
Columns: 17
